# Практикум 14. Вопросно-ответные системы (QA) и плотный поиск (DPR)

В лаборатории 14 вы ранжировали пассажи по вопросу, смотрели на двумерное «плотное» пространство, на матрицу B×B с in-batch negatives и на выбор ответа в тексте. Здесь повторим всё это в коде:

1. напишем BM25 с нуля и сравним его с TF-IDF из scikit-learn;
2. измерим качество поиска метриками Recall@k и MRR;
3. обучим крошечный двойной энкодер (dual encoder) на numpy с in-batch negatives — так обучают DPR;
4. сравним разреженный и плотный поиск по вопросам и попробуем гибрид;
5. напишем простого «читателя», который выделяет ответ в найденном пассаже.

*Практикум к лаборатории 14 курса «Алгоритмы NLP». Запускайте ячейки сверху вниз: Shift+Enter или «Среда выполнения → Выполнить всё». Чтобы сохранить свои ответы, сделайте «Файл → Сохранить копию на Диске».*

## 1. Данные: база знаний и вопросы

Открытая вопросно-ответная система (open-domain QA) обычно состоит из двух частей: **retriever** (поисковик) находит в большой коллекции несколько подходящих пассажей, **reader** (читатель) выделяет в них ответ. Наша коллекция — 18 коротких пассажей с простыми фактами. У каждого тестового вопроса известен правильный пассаж и ответ.

Отдельно есть обучающие пары «вопрос → пассаж» для плотного энкодера (раздел 3). Они сформулированы иначе, чем тестовые вопросы, иначе проверка была бы нечестной.

In [ ]:
# 18 коротких пассажей: это наша «база знаний»
passages = [
    "Ташкент — столица Узбекистана и самый большой город страны. В нём живёт около трёх миллионов человек.",
    "Самарканд — один из древнейших городов Средней Азии. В центре города находится площадь Регистан с тремя медресе.",
    "Улугбек построил в Самарканде обсерваторию в 1420-х годах. В его звёздном каталоге больше тысячи звёзд.",
    "Алишер Навои — поэт и мыслитель XV века. Он родился в 1441 году в Герате.",
    "Аль-Хорезми — математик IX века родом из Хорезма. От его имени произошло слово «алгоритм».",
    "Ибн Сина родился около 980 года недалеко от Бухары. Его «Канон врачебной науки» несколько веков служил главным учебником медицины.",
    "Аральское море когда-то было одним из крупнейших озёр мира. Оно сильно обмелело, потому что воду рек забрали на орошение полей.",
    "Амударья и Сырдарья — две главные реки Средней Азии. Исторически обе реки питали Аральское море.",
    "При нормальном атмосферном давлении вода кипит при 100 градусах Цельсия. В горах давление ниже, и вода закипает при более низкой температуре.",
    "Растения получают энергию из солнечного света. Этот процесс называется фотосинтезом, и при нём выделяется кислород.",
    "Луна — естественный спутник Земли. Свет от Луны доходит до Земли чуть больше чем за одну секунду.",
    "Свет от Солнца доходит до Земли примерно за 8 минут. Солнце — ближайшая к нам звезда.",
    "Узбекский язык относится к тюркской семье языков. С 1993 года для него официально принят латинский алфавит.",
    "Русский язык — восточнославянский язык индоевропейской семьи. На нём пишут кириллицей.",
    "Скоростной поезд, который связывает Ташкент с Самаркандом и Бухарой, называется «Афросиаб». Дорога от Ташкента до Самарканда занимает около двух часов.",
    "Исторический центр Бухары входит в список Всемирного наследия ЮНЕСКО. Минарет Калян построен в XII веке.",
    "Хива находится в Хорезмской области. Её внутренний город называется Ичан-Кала и окружён крепостной стеной.",
    "Компьютер хранит данные в двоичном коде, то есть с помощью нулей и единиц.",
]

# тестовые вопросы: (вопрос, номер правильного пассажа, ответ)
questions = [
    ("Сколько человек живёт в столице Узбекистана?", 0, "трёх миллионов"),
    ("Где родился Алишер Навои?", 3, "Герате"),
    ("Какой газ выделяется при фотосинтезе?", 9, "кислород"),
    ("За сколько минут солнечный свет долетает до Земли?", 11, "8"),
    ("При скольких градусах кипит вода?", 8, "100"),
    ("В каком году родился Авиценна?", 5, "980"),
    ("Как называется скоростной поезд между Ташкентом и Самаркандом?", 14, "Афросиаб"),
    ("Кто построил обсерваторию в Самарканде?", 2, "Улугбек"),
    ("Как называется внутренний город Хивы?", 16, "Ичан-Кала"),
    ("Какой алфавит используют для узбекского языка?", 12, "латинский"),
    ("Кто автор «Канона врачебной науки»?", 5, "Ибн Сина"),
    ("В каком веке жил аль-Хорезми?", 4, "IX"),
    ("Почему высохло Аральское море?", 6, "орошение"),
]

# обучающие пары (вопрос, пассаж) для плотного энкодера: по 2 на пассаж, вопросы не совпадают с тестовыми
train_questions = [
    ("Какой самый большой город Узбекистана?", 0), ("Сколько жителей в Ташкенте?", 0),
    ("Что находится в центре Самарканда?", 1), ("Сколько медресе на площади Регистан?", 1),
    ("Сколько звёзд в каталоге Улугбека?", 2), ("Когда в Самарканде появилась обсерватория?", 2),
    ("Кем был Алишер Навои?", 3), ("В каком веке жил Навои?", 3),
    ("Откуда родом аль-Хорезми?", 4), ("От чьего имени произошло слово алгоритм?", 4),
    ("Где родился Авиценна?", 5), ("Какую книгу по медицине написал Ибн Сина?", 5),
    ("Почему обмелело Аральское море?", 6), ("Куда ушла вода Аральского моря?", 6),
    ("Какие реки главные в Средней Азии?", 7), ("Какие реки питали Аральское море?", 7),
    ("Почему в горах вода закипает раньше?", 8), ("От чего зависит температура кипения воды?", 8),
    ("Откуда растения берут энергию?", 9), ("Что такое фотосинтез?", 9),
    ("Что такое Луна?", 10), ("Сколько идёт свет от Луны до Земли?", 10),
    ("Какая звезда ближе всего к Земле?", 11), ("Сколько времени идёт солнечный свет?", 11),
    ("К какой семье относится узбекский язык?", 12), ("С какого года узбеки пишут латиницей?", 12),
    ("Каким алфавитом пишут по-русски?", 13), ("К какой группе относится русский язык?", 13),
    ("Сколько ехать на поезде из Ташкента в Самарканд?", 14), ("Какой поезд ходит в Бухару?", 14),
    ("Когда построен минарет Калян?", 15), ("Какой город входит в список ЮНЕСКО?", 15),
    ("Где находится Хива?", 16), ("Что окружает Ичан-Калу?", 16),
    ("Как компьютер хранит данные?", 17), ("Что такое двоичный код?", 17),
]
print(len(passages), "пассажей,", len(questions), "тестовых вопросов,", len(train_questions), "обучающих пар")

Для поиска текст превращаем в список терминов: нижний регистр, без стоп-слов (в том числе вопросительных слов «кто», «где», «сколько»). Второй вариант — ещё и стемминг (Snowball из `nltk`), чтобы «Самарканде» и «Самарканда» совпали.

In [ ]:
import re
import numpy as np
from nltk.stem.snowball import SnowballStemmer   # стеммер Snowball входит в nltk, скачивать ничего не нужно

TOKEN_RE = r"\d+|[^\W\d_]+(?:-[^\W\d_]+)*"
STOP = set("""и в во на с со к ко о об от до из у за по при для а но же ли не что как кто где когда
какой какая какое какие каком какую какого какому скольких сколько почему это этот тот он она оно
они его её ее их нём нем ней него то есть был была было были чем чего х
называется""".split())
stemmer = SnowballStemmer("russian")

def tokenize(text):
    return re.findall(TOKEN_RE, text)

def norm(w):
    return w.lower().replace("ё", "е")

def stem(w):
    return stemmer.stem(norm(w))

def analyze_words(text):
    # слова в нижнем регистре без стоп-слов
    return [norm(w) for w in tokenize(text) if norm(w) not in STOP]

def analyze(text):
    # то же, но каждое слово обрезано стеммером до основы
    return [stem(w) for w in tokenize(text) if norm(w) not in STOP]

print(analyze_words(passages[2]))
print(analyze(passages[2]))
print(analyze(questions[7][0]))

## 2. Разреженный поиск: BM25 с нуля

BM25 суммирует вклады терминов вопроса $q$, которые встречаются в пассаже $d$:

$$\text{BM25}(q,d)=\sum_{t\in q}\text{idf}(t)\cdot\frac{tf(t,d)\,(k_1+1)}{tf(t,d)+k_1\left(1-b+b\frac{|d|}{\text{avgdl}}\right)},\qquad \text{idf}(t)=\ln\left(1+\frac{N-df(t)+0.5}{df(t)+0.5}\right)$$

Редкий термин (малое $df$) получает большой вес. Повтор термина помогает, но с насыщением (параметр $k_1$). Длинные пассажи штрафуются (параметр $b$). Ниже — вклад каждого термина вопроса в три лучших пассажа.

In [ ]:
import math
from collections import Counter

class BM25:
    def __init__(self, texts, analyzer, k1=1.5, b=0.75):
        self.analyzer, self.k1, self.b = analyzer, k1, b
        self.docs = [Counter(analyzer(t)) for t in texts]          # частоты терминов в каждом пассаже
        self.doc_len = np.array([sum(d.values()) for d in self.docs])
        self.avgdl = self.doc_len.mean()
        self.N = len(texts)
        self.df = Counter(t for d in self.docs for t in d)          # в скольких пассажах встречается термин

    def idf(self, t):
        return math.log(1 + (self.N - self.df[t] + 0.5) / (self.df[t] + 0.5))

    def term_score(self, t, i):
        tf = self.docs[i][t]
        norm_len = 1 - self.b + self.b * self.doc_len[i] / self.avgdl
        return self.idf(t) * tf * (self.k1 + 1) / (tf + self.k1 * norm_len)

    def scores(self, query):
        terms = self.analyzer(query)
        return np.array([sum(self.term_score(t, i) for t in terms) for i in range(self.N)])

bm25 = BM25(passages, analyze)
q = questions[7][0]
s = bm25.scores(q)
top = np.argsort(-s, kind="stable")[:3]
print(q)
print(f"{'термин':<14}{'df':>4}{'idf':>7}" + "".join(f"{'p' + str(i):>9}" for i in top))
for t in dict.fromkeys(analyze(q)):
    row = "".join(f"{bm25.term_score(t, i):9.2f}" for i in top)
    print(f"{t:<14}{bm25.df[t]:>4}{bm25.idf(t):7.2f}{row}")
print(f"{'сумма BM25':<25}" + "".join(f"{s[i]:9.2f}" for i in top))
print()
for i in top:
    print(f"p{i} ({s[i]:.2f}): {passages[i]}")

### Метрики поиска

Для каждого вопроса найдём **ранг** правильного пассажа в выдаче (1 — первый). **Recall@k** — доля вопросов, у которых правильный пассаж попал в первые $k$. **MRR** (mean reciprocal rank) — среднее $1/\text{ранг}$: 1.0, если всегда первый; 0.5, если всегда второй.


### То же в scikit-learn: TF-IDF и косинус

`TfidfVectorizer` строит вектор TF-IDF для каждого пассажа и нормирует его на длину 1. Тогда скалярное произведение вектора вопроса и вектора пассажа равно косинусу. Подставим наш анализатор, чтобы сравнение было честным.

In [ ]:
gold = np.array([g for _, g, _ in questions])

def ranks_from_scores(S, gold):
    # S: матрица оценок (вопросы × пассажи)
    ranks = []
    for s, g in zip(S, gold):
        order = np.argsort(-s, kind="stable")
        ranks.append(int(np.where(order == g)[0][0]) + 1)
    return np.array(ranks)

def metrics(ranks):
    return {"R@1": np.mean(ranks <= 1), "R@3": np.mean(ranks <= 3), "MRR": np.mean(1 / ranks)}

def fmt(m):
    return "  ".join(f"{k}={v:.2f}" for k, v in m.items())

results = {}
for name, an in [("слова", analyze_words), ("основы", analyze)]:
    model = BM25(passages, an)
    S = np.array([model.scores(q) for q, _, _ in questions])
    r = ranks_from_scores(S, gold)
    results["BM25, " + name] = r
    print(f"BM25, {name:<14} ранги: {r}   {fmt(metrics(r))}")


from sklearn.feature_extraction.text import TfidfVectorizer

for name, an in [("слова", analyze_words), ("основы", analyze)]:
    vec = TfidfVectorizer(analyzer=an)            # используем тот же анализатор, что и в BM25
    P = vec.fit_transform(passages)               # строки нормированы: длина вектора = 1
    Qm = vec.transform([q for q, _, _ in questions])
    S = (Qm @ P.T).toarray()                      # косинусы вопрос × пассаж
    r = ranks_from_scores(S, gold)
    results["TF-IDF, " + name] = r
    print(f"TF-IDF, {name:<13} ранги: {r}   {fmt(metrics(r))}")

import matplotlib.pyplot as plt
names = list(results)
vals = np.array([[metrics(results[n])[k] for k in ["R@1", "R@3", "MRR"]] for n in names])
fig, ax = plt.subplots(figsize=(8, 3))
for j, k in enumerate(["R@1", "R@3", "MRR"]):
    ax.bar(np.arange(len(names)) + 0.25 * (j - 1), vals[:, j], width=0.25, label=k)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, fontsize=8)
ax.set_ylim(0, 1.05)
ax.set_title("Разреженный поиск: качество на тестовых вопросах")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 3. Плотный поиск: двойной энкодер и in-batch negatives

В DPR (Dense Passage Retrieval) вопрос и пассаж кодируются **двумя разными** энкодерами $E_Q$ и $E_P$ в векторы одной размерности, сходство — скалярное произведение $\text{sim}(q,p)=E_Q(q)\cdot E_P(p)$. Векторы всех пассажей считаются заранее, поэтому поиск быстрый.

Обучение: в батче $B$ пар (вопрос$_i$, пассаж$_i$). Матрица $S=QP^\top$ размера $B\times B$: на диагонали положительные пары, вне диагонали — пассажи других вопросов, они служат отрицательными примерами (**in-batch negatives**). Функция потерь — кросс-энтропия по строкам:

$$L=-\frac{1}{B}\sum_i \log\frac{e^{S_{ii}}}{\sum_j e^{S_{ij}}}$$

Вместо BERT наш энкодер — линейная проекция: признаки (TF-IDF по символьным n-граммам, 2–4 символа) умножаются на матрицу $W_Q$ или $W_P$ размера $D\times 64$. Обе матрицы начинаем с одной и той же случайной матрицы. Случайная проекция примерно сохраняет скалярные произведения, поэтому до обучения модель похожа на обычный символьный TF-IDF.

In [ ]:
# признаки: TF-IDF по символьным n-граммам (2–4 символа внутри слов)
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True)
char_vec.fit(passages + [q for q, _ in train_questions])
X_pass = char_vec.transform(passages).toarray()
X_train = char_vec.transform([q for q, _ in train_questions]).toarray()
y_train = np.array([p for _, p in train_questions])
X_test = char_vec.transform([q for q, _, _ in questions]).toarray()
D = X_pass.shape[1]
print("число признаков D:", D)

def batch_loss(Wq, Wp, Xq, Xp):
    # Xq, Xp: признаки B вопросов и B их пассажей (строка i ↔ строка i)
    Q, P = Xq @ Wq, Xp @ Wp            # линейные энкодеры
    S = Q @ P.T                         # B×B: все вопросы × все пассажи батча
    A = np.exp(S - S.max(axis=1, keepdims=True))
    A /= A.sum(axis=1, keepdims=True)   # softmax по строкам
    B = len(S)
    loss = -np.mean(np.log(np.diag(A)))
    dS = (A - np.eye(B)) / B            # dL/dS
    return loss, Xq.T @ (dS @ P), Xp.T @ (dS.T @ Q), S, A

rng = np.random.default_rng(0)
d = 64
W0 = rng.normal(0, 1 / np.sqrt(d), size=(D, d))    # случайная проекция D -> 64
Wq, Wp = W0.copy(), W0.copy()

demo_p = np.array([0, 3, 5, 9, 11, 14])                       # демонстрационный батч: 6 пассажей и по одному обучающему вопросу
demo_q = np.array([np.where(y_train == p)[0][0] for p in demo_p])
loss, gq, gp, S, A = batch_loss(Wq, Wp, X_train[demo_q], X_pass[demo_p])
np.set_printoptions(precision=2, suppress=True)
for qi in demo_q:
    print("  q:", train_questions[qi][0])
print("матрица сходства S (строки — вопросы, столбцы — пассажи):\n", S)
print("вероятность правильного пассажа (диагональ softmax):", np.diag(A))
print(f"loss = {loss:.3f}   (случайное угадывание: ln {len(demo_p)} = {np.log(len(demo_p)):.3f})")

# проверка градиента: аналитический против численного (конечная разность)
j = int(np.nonzero(X_train[demo_q[0]])[0][0])
eps = 1e-5
Wq2 = Wq.copy(); Wq2[j, 0] += eps
numeric = (batch_loss(Wq2, Wp, X_train[demo_q], X_pass[demo_p])[0] - loss) / eps
print(f"градиент по Wq[j, 0]: аналитический {gq[j, 0]:.6f}, численный {numeric:.6f}")

Обучаем градиентным спуском: в каждом шаге берём 8 **разных** пассажей и по одному обучающему вопросу к каждому. Если два вопроса в батче относятся к одному пассажу, «отрицательный» пример окажется правильным. Посмотрим на loss и на ту же матрицу 6×6 до и после обучения: диагональ должна стать яркой.

In [ ]:
def dense_scores(Xq, Wq, Wp):
    return (Xq @ Wq) @ (X_pass @ Wp).T

r_char = ranks_from_scores(X_test @ X_pass.T, gold)
r_before = ranks_from_scores(dense_scores(X_test, Wq, Wp), gold)
print("char TF-IDF (без проекции):  ", fmt(metrics(r_char)))
print("плотная модель до обучения:   ", fmt(metrics(r_before)))

S_before = batch_loss(Wq, Wp, X_train[demo_q], X_pass[demo_p])[3]
B, lr, steps = 8, 0.3, 300
history = []
for step in range(steps + 1):
    pids = rng.choice(len(passages), size=B, replace=False)        # 8 разных пассажей: без «ложных отрицательных»
    qids = [rng.choice(np.where(y_train == p)[0]) for p in pids]
    loss, gq, gp, _, _ = batch_loss(Wq, Wp, X_train[qids], X_pass[pids])
    Wq -= lr * gq
    Wp -= lr * gp
    history.append(loss)
    if step % 50 == 0:
        print(f"шаг {step:3d}   loss = {loss:.3f}")
S_after = batch_loss(Wq, Wp, X_train[demo_q], X_pass[demo_p])[3]

r_dense = ranks_from_scores(dense_scores(X_test, Wq, Wp), gold)
results["плотная (dual encoder)"] = r_dense
print()
print("после обучения, обучающие вопросы:", fmt(metrics(ranks_from_scores(dense_scores(X_train, Wq, Wp), y_train))))
print("после обучения, тестовые вопросы: ", fmt(metrics(r_dense)))

fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
ax[0].plot(history, lw=0.8)
ax[0].set_title("loss по шагам")
ax[0].set_xlabel("шаг")
labels = ["p" + str(p) for p in demo_p]
for a, M, t in [(ax[1], S_before, "S до обучения"), (ax[2], S_after, "S после обучения")]:
    a.imshow(M, cmap="viridis")
    a.set_xticks(range(len(labels))); a.set_xticklabels(labels)
    a.set_yticks(range(len(labels))); a.set_yticklabels(["q" + str(p) for p in demo_p])
    a.set_title(t)
plt.tight_layout()
plt.show()

### Где выигрывает каждый метод

Сравним ранги правильного пассажа по каждому вопросу. Плотная модель училась на парафразах: например, в обучающих данных было «Где родился Авиценна?» → пассаж про Ибн Сину. В самом пассаже слова «Авиценна» нет, BM25 его не найдёт. Зато BM25 точно ловит редкие слова и числа, если они есть в вопросе. Поэтому на практике часто используют **гибрид**: сумму нормированных оценок обоих методов.

Обратите внимание: на тестовых вопросах обученная модель не обязательно лучше исходного символьного TF-IDF. 36 обучающих пар — очень мало, и модель отчасти просто запоминает их (на обучающих вопросах MRR = 1.00). Не делайте выводов о «победителе» по 13 вопросам: разница в один вопрос — это шум. Настоящий DPR обучают на десятках тысяч пар и добавляют «трудные» отрицательные примеры из выдачи BM25.

In [ ]:
r_bm25 = results["BM25, основы"]
print(f"{'BM25':>5} {'плотн.':>7}  вопрос")
for (q, g, _), a, b in zip(questions, r_bm25, r_dense):
    mark = "  <- BM25" if a < b else ("  <- плотн." if b < a else "")
    print(f"{a:>5} {b:>7}  {q}{mark}")

# гибрид: сумма оценок, нормированных по каждому вопросу (z-оценки)
S_bm = np.array([bm25.scores(q) for q, _, _ in questions])
S_de = dense_scores(X_test, Wq, Wp)
z = lambda M: (M - M.mean(axis=1, keepdims=True)) / (M.std(axis=1, keepdims=True) + 1e-9)
results["гибрид"] = ranks_from_scores(z(S_bm) + z(S_de), gold)
print()
for name in ["BM25, основы", "TF-IDF, основы", "плотная (dual encoder)", "гибрид"]:
    print(f"{name:<22} {fmt(metrics(results[name]))}")

### Как это выглядит на настоящей модели

`intfloat/multilingual-e5-small` — многоязычный энкодер предложений (118 млн параметров, вектор размерности 384, файл весов ≈470 МБ). Он один и тот же для вопросов и пассажей, но различает их по префиксам: к вопросу добавляют `"query: "`, к пассажу — `"passage: "`. Без префиксов качество заметно падает. Ячейка работает только в Colab с интернетом.

In [ ]:
# в Colab sentence-transformers уже установлен (иначе: !pip install -q sentence-transformers)
try:
    from sentence_transformers import SentenceTransformer
    e5 = SentenceTransformer("intfloat/multilingual-e5-small")      # ≈470 МБ, скачивается один раз
    # префиксы обязательны для моделей E5
    P_e5 = e5.encode(["passage: " + p for p in passages], normalize_embeddings=True)
    Q_e5 = e5.encode(["query: " + q for q, _, _ in questions], normalize_embeddings=True)
    print("векторы пассажей:", P_e5.shape)
    r_e5 = ranks_from_scores(Q_e5 @ P_e5.T, gold)
    print("multilingual-e5-small:", fmt(metrics(r_e5)))
    print("BM25 (основы):    ", fmt(metrics(results["BM25, основы"])))
    for (q, g, _), r in zip(questions, r_e5):
        print(f"  {r:>2}  {q}")
except Exception as e:
    print("Не удалось загрузить модель (нужен интернет в Colab):", e)

## 4. Читатель: выделяем ответ в пассаже

Нейронный читатель (BERT в DPR) для каждого токена предсказывает вероятность начала и конца ответа. Мы сделаем простой читатель на правилах, который показывает те же идеи:

1. **кандидаты** — слова пассажа, которых нет в вопросе (ответ обычно не повторяет вопрос), кроме стоп-слов;
2. **близость**: кандидат получает балл $\sum_j \text{idf}(w_j)/|i-j|$ по всем словам вопроса $w_j$, найденным в пассаже на позиции $j$;
3. **тип ответа** по вопросительному слову: «сколько», «в каком году» → число; «где» → место (слово с заглавной буквы после предлога места); «кто» → имя (слово с заглавной буквы). Если подходящие по типу кандидаты есть, выбираем только из них;
4. **фокус**: в вопросе «какой алфавит» ответ обычно стоит прямо перед словом «алфавит» («латинский алфавит»), а в вопросе «как называется…» — сразу после слова «называется». Такой кандидат получает +3.

Ответ — лучший кандидат, расширенный соседними кандидатами того же типа (до 3 слов: «Ибн Сина», «трёх миллионов»). Это грубая замена вероятностей начала и конца ответа.

In [ ]:
SPAN_RE = r"\d+|[^\W\d_]+(?:-[^\W\d_]+)*|[^\w\s]"     # как TOKEN_RE, плюс знаки препинания отдельными токенами
# тип ожидаемого ответа по вопросительному слову
TYPE_PATTERNS = [
    ("number", r"сколько|скольких|каком году|каком веке"),
    ("place",  r"^где\b|откуда"),
    ("person", r"^кто\b"),
    ("name",   r"как называется"),
]
NUMBER_RE = r"\d+|один|одна|два|две|двух|три|трёх|трех|четыре|пять|десять|сто|тысяч\w*|миллион\w*"
PLACE_PREP = {"в", "во", "из", "от", "у", "около"}

def is_place(toks, i):
    # место: слово с заглавной буквы после предлога места («в Герате», «от Бухары»)
    return i > 0 and toks[i - 1].lower() in PLACE_PREP

def question_focus(question):
    # «какой алфавит» -> (основа «алфав», +1): ответ стоит прямо ПЕРЕД словом «алфавит» («латинский алфавит»)
    # «как называется» -> (основа «называ», -1): ответ стоит прямо ПОСЛЕ «называется»
    ql = question.lower()
    if "как называется" in ql:
        return (stem("называется"), -1)
    m = re.search(r"\b(?:какой|какая|какое|какие|каком|какую|какого)\s+(\w+)", ql)
    return (stem(m.group(1)), +1) if m else None

def is_cap(tok):
    return tok[:1].isupper()

def answer_type(question):
    ql = question.lower()
    for typ, pattern in TYPE_PATTERNS:
        if re.search(pattern, ql):
            return typ
    return None

def token_type_ok(toks, i, typ):
    t = toks[i]
    if typ == "number":
        return bool(re.fullmatch(NUMBER_RE, t.lower()) or re.fullmatch(r"[IVXLC]+", t))
    if typ == "name":
        return True                  # название может быть любым словом
    if typ == "place":
        return is_cap(t) and is_place(toks, i)
    if typ == "person":
        return is_cap(t) and not is_place(toks, i)
    return True

def next_word(toks, i, step):
    # nearest word token to the left (step=-1) or right (step=+1), punctuation skipped
    k = i + step
    while 0 <= k < len(toks) and not toks[k][0].isalnum():
        k += step
    return k if 0 <= k < len(toks) else None

def read(question, passage, bm=bm25, verbose=False):
    toks = re.findall(SPAN_RE, passage)
    q_stems = set(analyze(question))
    typ = answer_type(question)
    focus = question_focus(question)          # (основа, сторона) или None
    match_pos = [j for j, t in enumerate(toks) if t[0].isalnum() and stem(t) in q_stems]
    # кандидаты: слова пассажа, которых нет в вопросе, без стоп-слов
    cand = [i for i, t in enumerate(toks)
            if t[0].isalnum() and norm(t) not in STOP and stem(t) not in q_stems]
    score = {}
    for i in cand:
        # близость к словам вопроса, найденным в пассаже, с весом idf
        prox = sum(bm.idf(stem(toks[j])) / abs(i - j) for j in match_pos)
        # кандидат прямо перед фокусным словом вопроса
        k = next_word(toks, i, focus[1]) if focus else None
        near_focus = k is not None and stem(toks[k]) == focus[0]
        score[i] = prox + (3.0 if near_focus else 0.0)
    typed = [i for i in cand if typ and token_type_ok(toks, i, typ)]
    pool = typed if typed else cand                  # если есть кандидаты нужного типа — выбираем только из них
    best = max(pool, key=lambda i: score[i])
    # расширяем ответ соседними кандидатами того же типа (до 3 слов)
    span = [best]
    if typ is not None:
        for step in (-1, 1):
            k = best + step
            while k in score and k in pool and len(span) < 3:
                span.append(k); k += step
    span.sort()
    answer = " ".join(toks[i] for i in span)
    if verbose:
        print(f"тип ответа: {typ}, фокус: {focus}, слова вопроса в пассаже: {[toks[j] for j in match_pos]}")
        for i in sorted(pool, key=lambda i: -score[i])[:4]:
            print(f"    {toks[i]:<14} {score[i]:5.2f}")
    return answer, toks, score, span

for k in [1, 9, 10]:
    q, g, ans = questions[k]
    print(q)
    pred = read(q, passages[g], verbose=True)[0]
    print("  ->", pred, "  (верно:", ans + ")\n")

Баллы кандидатов на одном пассаже (серые столбики — слова вопроса и стоп-слова, оранжевые — выбранный ответ):

In [ ]:
q, g, ans = questions[1]
pred, toks, score, span = read(q, passages[g])
x = np.arange(len(toks))
h = [score.get(i, 0) for i in x]
colors = ["#D9822B" if i in span else ("#2B55C9" if i in score else "#BBBBBB") for i in x]
fig, ax = plt.subplots(figsize=(11, 3))
ax.bar(x, h, color=colors)
ax.set_xticks(x)
ax.set_xticklabels(toks, rotation=70, fontsize=8)
ax.set_title("баллы кандидатов: " + q)
plt.tight_layout()
plt.show()

### Вся система: BM25 → читатель

Берём первый пассаж из выдачи BM25 и запускаем читатель. Ответ засчитываем, если найденный фрагмент содержит правильный ответ или содержится в нём. Посмотрите, где ошибается поиск, а где читатель. Вопрос «почему…» требует ответа-причины (целого предложения), а наш читатель умеет выделять только короткие фрагменты.

In [ ]:
# первый пассаж из выдачи BM25 -> читатель
correct = 0
for q, g, ans in questions:
    top1 = int(np.argmax(bm25.scores(q)))
    pred = read(q, passages[top1])[0]
    ok = ans.lower() in pred.lower() or pred.lower() in ans.lower()
    correct += ok
    print(f"{'+' if ok else '-'} p{top1:<3} {pred:<22} ({ans})  {q}")
print(f"\nверных ответов: {correct} / {len(questions)}")

## 5. Задания

1. **Параметры BM25.** Посчитайте MRR для $k_1 \in \{0.5, 1.2, 2.0\}$ и $b \in \{0, 0.75, 1\}$ (9 комбинаций, таблица). Что означает $b=0$? Почему на наших коротких пассажах разница маленькая?
2. **Свои вопросы.** Придумайте 4 новых вопроса к пассажам: 2 с синонимами слов пассажа (парафраз) и 2 с редким именем или числом. Сравните ранги BM25 и плотной модели.
3. **Размер батча.** Обучите двойной энкодер заново (от `W0`) с $B = 2, 4, 8, 16$ при одинаковом числе шагов. Как меняются итоговый loss и MRR на тесте? Почему при $B=2$ задача слишком лёгкая?
4. **Гибрид с весом.** Вместо простой суммы используйте $\alpha\cdot z(\text{BM25})+(1-\alpha)\cdot z(\text{dense})$ и найдите лучшее $\alpha$ из сетки 0, 0.1, …, 1. Почему подбирать $\alpha$ на тех же тестовых вопросах — плохая практика?
5. **Читатель.** Добавьте в `TYPE_PATTERNS` тип для вопросов «когда» (ответ — число из 3–4 цифр) и проверьте его на вопросах «Когда родился Ибн Сина?» и «Когда построен минарет Калян?». Какие ошибки остались?

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь